# Reorganization metrics (single patient)

Compute phase-by-phase distance matrices from cached LRG results for one patient and one band.

**Legacy notebooks merged:**
- NEW_distance_of_distances.ipynb
- NEW_standard_disXpat.ipynb
- NEW_tree_measures_pat_comparison.ipynb
- NEW_ultrametric_quantile_rmse_pat_comparison.ipynb
- NEW_ultrametric_rank_correlation_pat_comparison.ipynb
- NEW_ultrametric_scaled_distance_pat_comparison.ipynb
- distance_of_distances.ipynb

In [ ]:
%matplotlib inline
from lrgsglib.config.funcs import move_to_rootf
move_to_rootf(pathname="lrg_eegfc")
from lrg_eegfc.notebook import *

In [ ]:
DATA_ROOT = Path('data/stereoeeg_patients')
patients = list_patients(DATA_ROOT)
assert patients, 'No patients found under data/stereoeeg_patients'

patient = patients[0]
band = BRAIN_BANDS_NAMES[2]  # alpha
phases = list(PHASE_LABELS)
fc_method = 'msc'

results_by_phase = {
    phase: load_lrg_result(patient, phase, band, fc_method)
    for phase in phases
}
missing = [p for p, r in results_by_phase.items() if r is None]
assert not missing, f'Missing LRG cache for phases: {missing}'
results_by_phase

In [ ]:
from lrg_eegfc.utils.metrics.reorganization import (
    build_metric_specs,
    compute_metric_matrix,
    compute_cluster_labels,
    compute_cluster_swap_matrix,
    compute_ari_matrix,
)

metric_specs = build_metric_specs(distance_metric='euclidean')
metric_matrices = {
    key: compute_metric_matrix(phases, results_by_phase, spec['fn'])
    for key, spec in metric_specs.items()
}

labels_by_phase = {
    phase: compute_cluster_labels(
        results_by_phase[phase].linkage_matrix,
        results_by_phase[phase].optimal_threshold,
    )
    for phase in phases
}
metric_matrices['cluster_swap'] = compute_cluster_swap_matrix(phases, labels_by_phase)
metric_matrices['cluster_ari'] = compute_ari_matrix(phases, labels_by_phase)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

phase_ticks = np.arange(len(phases))

for key, matrix in metric_matrices.items():
    fig, ax = plt.subplots(figsize=(4, 4))
    im = ax.imshow(matrix, cmap='viridis')
    ax.set_xticks(phase_ticks)
    ax.set_yticks(phase_ticks)
    ax.set_xticklabels(phases, rotation=45, ha='right')
    ax.set_yticklabels(phases)
    ax.set_title(key)
    fig.colorbar(im, ax=ax, shrink=0.8)
    plt.show()